<a href="https://colab.research.google.com/github/usman-stack-322/flyrank-ml-internship-v2/blob/main/work/notebooks/w04_signal_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/usman-stack-322/flyrank-ml-internship-v2/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Distributions


The observed distributions show substantial variation across search demand, search performance, ranking position, content age, and recent trend. Some fields have large differences between their median and maximum values, indicating potential heavy tails and a smaller number of pages with unusually high values. These distributions should be considered when interpreting simple signal thresholds.


In [10]:
# Clone the FlyRank internship repository
!git clone https://github.com/flyrank-bih/flyrank-ml-internship-starter.git

fatal: destination path 'flyrank-ml-internship-starter' already exists and is not an empty directory.


In [11]:
# Load the FlyRank content refresh dataset

import pandas as pd
import numpy as np

file_path = "flyrank-ml-internship-starter/data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(file_path)

print("Dataset shape:", df.shape)
print("\nColumns:", df.columns.tolist())


# Inspect distributions of key content-opportunity signals

key_fields = [
    "search_volume",
    "impressions_90d",
    "clicks_90d",
    "ctr",
    "avg_position",
    "content_age_days",
    "trend_pct"
]

display(df[key_fields].describe().T)

Dataset shape: (30000, 44)

Columns: ['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


,count,mean,std,min,25%,50%,75%,max
search_volume,27532.0,158.882391,1518.270825,0.0,0.0,10.00,20.00,74000.0
impressions_90d,30000.0,5200.366300,16838.019547,1.0,81.0,731.00,3615.25,517715.0
clicks_90d,30000.0,16.097333,75.076958,0.0,0.0,1.00,7.00,4178.0
ctr,30000.0,0.510733,3.279162,0.0,0.0,0.07,0.29,100.0
avg_position,30000.0,16.342380,15.216790,0.0,6.2,10.80,22.30,245.0
content_age_days,30000.0,256.167800,132.707930,90.0,132.0,236.00,333.00,564.0
trend_pct,26612.0,-4.785969,473.861780,-100.0,-62.6,-33.50,0.00,44900.0


### Signal Test Results

**Signal #1 — High impressions + low CTR: OPPOSITE**

The observed median CTR for high-impression pages was 0.18, compared with 0.07 for the overall dataset. In this slice, high impressions were associated with a higher observed median CTR rather than a lower one. The proposed assumption is therefore opposite to the observed result.

**Signal #2 — Position 8–15: CONFIRMED**

There were 7,532 observed pages with average positions between 8 and 15. Their median CTR was 0.11, compared with 0.05 for pages outside this range. This supports the directional idea that pages in positions 8–15 can be a useful segment for further content-opportunity analysis, although the comparison alone does not prove that refreshing these pages will improve performance.

**Signal #3 — High search volume + low clicks: MIXED**

The high-search-volume group had a median of 1.0 clicks, which was the same as the overall median of 1.0 clicks. The median comparison does not show a clear directional difference, so this signal is treated as mixed rather than confirmed.


In [12]:
# Signal test #1: high impressions + low CTR

impression_cutoff = df["impressions_90d"].median()

high_impression = df[df["impressions_90d"] >= impression_cutoff]

print("High-impression pages:", len(high_impression))
print("Overall median CTR:", df["ctr"].median())
print("High-impression median CTR:", high_impression["ctr"].median())

High-impression pages: 15007
Overall median CTR: 0.07
High-impression median CTR: 0.18


### Signal Test #2 — Position 8–15

Hypothesis: Pages ranking around positions 8–15 may have an opportunity to improve because they are already visible in search but are not yet near the top results.

I measure the number of pages in this position range and compare their observed CTR with pages outside the range.

**Verdict:** The result is based on the measured data.


In [13]:
# Signal test #2: pages ranking in positions 8–15

position_group = df[
    (df["avg_position"] >= 8) &
    (df["avg_position"] <= 15)
]

outside_group = df[
    ~df.index.isin(position_group.index)
]

print("Pages in positions 8–15:", len(position_group))
print("Median CTR for positions 8–15:", position_group["ctr"].median())
print("Median CTR outside positions 8–15:", outside_group["ctr"].median())

Pages in positions 8–15: 7532
Median CTR for positions 8–15: 0.11
Median CTR outside positions 8–15: 0.05


### Signal Test #3 — High Search Volume + Low Clicks

Hypothesis: Pages associated with high search volume but relatively low observed clicks may represent potential content opportunities.

I compare the click volume of high-search-volume pages with the overall median clicks.

**Verdict:** The result is based on the measured comparison.


In [14]:
# Signal test #3: high search volume + low clicks

search_cutoff = df["search_volume"].median()

high_search = df[df["search_volume"] >= search_cutoff]

print("High-search-volume pages:", len(high_search))
print("Overall median clicks:", df["clicks_90d"].median())
print("High-search-volume median clicks:", high_search["clicks_90d"].median())


High-search-volume pages: 16451
Overall median clicks: 1.0
High-search-volume median clicks: 1.0


## 3. Flag-Linked Test — Position 8–15

One FlyRank opportunity flag relies on the assumption that pages ranking around positions 8–15 are useful candidates for content improvement.

I test this assumption by comparing the observed CTR of pages in positions 8–15 with pages outside that range. A higher CTR in the 8–15 group would support the idea that these pages already receive meaningful search visibility and may be a useful segment for decision-support.

**Verdict: CONFIRMED, with caution.**

The observed CTR is higher for the 8–15 group, but this is a directional association and does not establish that refreshing these pages will cause higher performance.


In [15]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Flag-linked test: does the position 8–15 assumption hold?

flag_group = df[
    (df["avg_position"] >= 8) &
    (df["avg_position"] <= 15)
]

non_flag_group = df[
    ~df.index.isin(flag_group.index)
]

flag_ctr = flag_group["ctr"].median()
non_flag_ctr = non_flag_group["ctr"].median()

print("Pages flagged by position 8–15:", len(flag_group))
print("Median CTR of flagged pages:", flag_ctr)
print("Median CTR of non-flagged pages:", non_flag_ctr)

if flag_ctr > non_flag_ctr:
    print("\nVerdict: CONFIRMED")
elif flag_ctr < non_flag_ctr:
    print("\nVerdict: OPPOSITE")
else:
    print("\nVerdict: MIXED")


Pages flagged by position 8–15: 7532
Median CTR of flagged pages: 0.11
Median CTR of non-flagged pages: 0.05

Verdict: CONFIRMED


## 4.What This Means in Practice

The observed data suggests that pages ranking in positions 8–15 are a useful segment for content-opportunity decision-support because their median CTR was higher than pages outside this range. However, the high-impression/low-CTR assumption was opposite to the observed data, while the high-search-volume/low-clicks signal was mixed. A content team should therefore use the position signal as a directional prioritization input rather than treating any single flag as proof that a refresh will improve performance.


In [16]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


# Final practical check for the signal audit

print("Signal audit summary:")
print("- High impressions + low CTR: OPPOSITE")
print("- Position 8–15: CONFIRMED")
print("- High search volume + low clicks: MIXED")
print("- Flag-linked position test: CONFIRMED")


Signal audit summary:
- High impressions + low CTR: OPPOSITE
- Position 8–15: CONFIRMED
- High search volume + low clicks: MIXED
- Flag-linked position test: CONFIRMED


## Self-check

Before you submit, confirm each line honestly:

- [X] Every section above is filled — markdown thinking AND the code that backs it
- [X] The notebook runs top to bottom with no errors (Runtime → Run all)
- [X] No client names, URLs, or private queries anywhere
- [X] My claims use careful words: observed, measured, directional, decision-support
- [X] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.